# Plate with circular hole — Kirsch benchmark

Section 2.4.3 of Cornejo Fuentes (INSA Lyon, 2024).  
Infinite plate with a circular hole of radius $R_{in}=1$ under uniaxial tension $T_x=1$ in the $x$-direction.  
The analytic stress field was derived by **Kirsch (1898)** — *Die Theorie der Elastizität und die Bedürfnisse
der Festigkeitslehre*, Z. Vereines Deutscher Ingenieure, **42**, 797–807.  
The domain is the quarter annulus $r \in [R_{in}, R_{ex}]$, $\theta \in [0,\pi/2]$.

**Boundary conditions**

| Edge | BC type | Condition |
|---|---|---|
| Inner arc $r = R_{in}$ | Natural (traction-free) | $\boldsymbol{\sigma}\cdot\mathbf{n} = \mathbf{0}$ |
| Outer arc $r = R_{ex}$ | Neumann — Python callable | exact Kirsch traction $\mathbf{g}_{ext}$ |
| Bottom edge $\theta = 0$ | Dirichlet — symmetry | $u_y = 0$ |
| Left edge $\theta = \pi/2$ | Dirichlet — symmetry | $u_x = 0$ |

**Material**: plane stress, $E = 10^3$, $\nu = 0.3$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse.linalg as spla

from yeti_iga.future.bspline import (
    BSpline, BSplineSurface, ControlPointManager,
    Patch, PRefiner, SubdivisionRefiner,
    GlobalDOFManager, PatchDOFManager,
    PatchIntegrator, PatchEvaluator, MaterialProperties, IGABasis1D,
    Traction, ScalarLocalOperator,
)
from yeti_iga.future.plotting import plot_patches_2d

## Geometry — exact NURBS quarter ring

The quarter annulus is parametrised as in notebook 09:
- **u-direction**: angular ($\theta$ from $0$ to $\pi/2$), B-spline degree 2
- **v-direction**: radial ($r$ from $R_{in}$ to $R_{ex}$), B-spline degree 1

Corner control points carry NURBS weight $w = 1/\sqrt{2}$ to represent the arcs exactly.

In [ ]:
Rin  = 1.0
Rex  = 2.0   # Rex=2 keeps the analytic singularity of 1/r(v) at v=-1 (distance 1 from [0,1]),
              # well outside the parametric domain.  Rex=4 would push it to v=-1/3 (dist 1/3),
              # strongly extending the pre-asymptotic regime for high degrees (p≥4).
Tx   = 1.0   # far-field tension
E    = 1e3
nu   = 0.3

w_corner = 1.0 / np.sqrt(2.0)

mgr = ControlPointManager(dim=2)
# Inner arc (iv = 0): theta from 0 to pi/2
mgr.add_point([Rin, 0.0 ], w=1.0     )   # 0
mgr.add_point([Rin, Rin  ], w=w_corner)   # 1  (corner, activates NURBS)
mgr.add_point([0.0, Rin  ], w=1.0     )   # 2
# Outer arc (iv = 1)
mgr.add_point([Rex, 0.0 ], w=1.0     )   # 3
mgr.add_point([Rex, Rex  ], w=w_corner)   # 4
mgr.add_point([0.0, Rex  ], w=1.0     )   # 5

su = BSpline(2, np.array([0., 0., 0., 1., 1., 1.]))  # angular, degree 2
sv = BSpline(1, np.array([0., 0., 1., 1.]))            # radial,  degree 1

patch = Patch(BSplineSurface(su, sv), mgr, list(range(6)), [3, 2])
print(f'Initial mesh — {patch.n_cp} CPs, rational: {mgr.is_rational}')
plot_patches_2d(patch, show_control_points=True, title='Initial mesh (1 element)')

## k-refinement

k-refinement = degree elevation followed by knot insertion (no repeated internal knots).  
Here we elevate both directions to degree 3 (angular: 2→3, radial: 1→3), then subdivide
uniformly into 8×8 elements (`n_levels=3` → $2^3 = 8$ spans per direction).

In [ ]:
PRefiner(direction=0, n_elevations=1).refine(patch)       # angular: degree 2 → 3
PRefiner(direction=1, n_elevations=2).refine(patch)       # radial: degree 1 → 3
SubdivisionRefiner(direction=0, n_levels=3).refine(patch)
SubdivisionRefiner(direction=1, n_levels=3).refine(patch)

pu, pv = patch.tensor.components[0].degree, patch.tensor.components[1].degree
nu_cp, nv_cp = patch.local_shape[0], patch.local_shape[1]
print(f'After k-refinement: degree {pu}×{pv}, {nu_cp}×{nv_cp} = {patch.n_cp} CPs')
plot_patches_2d(patch, show_control_points=True,
                title=f'After k-refinement (degree {pu}×{pv}, {patch.n_cp} CPs)')

## DOF management and stiffness matrix

Two scalar DOFs (displacement components $u_x$, $u_y$) per control point.  
A new `Patch` object wrapping the same geometry data carries the `PatchDOFManager`.

In [ ]:
n_cp  = patch.n_cp
n_dof = 2 * n_cp

gm      = GlobalDOFManager([2] * n_cp)
pdm     = PatchDOFManager(2, list(range(n_cp)), gm)
patch_k = Patch(patch.tensor, patch.cp_manager,
                list(patch.global_indices), list(patch.local_shape), pdm)

su_k, sv_k = patch_k.tensor.components
basis_u = IGABasis1D.build(su_k, su_k.degree + 1)
basis_v = IGABasis1D.build(sv_k, sv_k.degree + 1)

mat  = MaterialProperties(E=E, nu=nu)
K    = PatchIntegrator(patch_k, basis_u, basis_v, mat).integrate_stiffness()

print(f'n_cp  = {n_cp},  n_dof = {n_dof}')
print(f'K : {K.shape}, nnz = {K.nnz}, symmetry error = {np.max(np.abs(K - K.T)):.2e}')

## Python-callable traction — exact Kirsch load on the outer arc

Subclassing `Traction` and overriding `evaluate(physical_point)` allows any position-dependent
traction to be passed directly to `PatchIntegrator.integrate_boundary_load()`.

The exact Kirsch traction at a point $(x, y)$ with polar coordinates $(r, \theta)$:

$$
g_1 = \frac{T_x}{2}\!\left(2\cos\theta - \frac{R_{in}^2}{r^2}(2\cos\theta+3\cos 3\theta)
      + \frac{3R_{in}^4}{r^4}\cos 3\theta\right)
$$
$$
g_2 = \frac{3T_x}{2}\sin 3\theta\!\left(\frac{R_{in}^4}{r^4} - \frac{R_{in}^2}{r^2}\right)
$$

In the NURBS parametrisation, the outer arc corresponds to the **v-direction max** edge:
`direction=1, side=1` (v is the radial direction, side 1 = outer).

In [ ]:
class KirschTraction(Traction):
    """Exact Kirsch traction at the outer arc of the quarter annulus."""

    def __init__(self, Rin, Tx=1.0):
        super().__init__()
        self.Rin = Rin
        self.Tx  = Tx

    def evaluate(self, pt):
        x, y  = pt[0], pt[1]
        r     = np.sqrt(x**2 + y**2)
        theta = np.arctan2(y, x)
        R2 = (self.Rin / r) ** 2
        R4 = (self.Rin / r) ** 4
        g1 = (self.Tx / 2) * (
            2 * np.cos(theta)
            - R2 * (2 * np.cos(theta) + 3 * np.cos(3 * theta))
            + 3 * R4 * np.cos(3 * theta)
        )
        g2 = (3 * self.Tx / 2) * np.sin(3 * theta) * (R4 - R2)
        return np.array([g1, g2])


kt = KirschTraction(Rin, Tx)
g_check = kt.evaluate(np.array([Rex, 0.0]))
print(f'g_ext at (Rex, 0) = {g_check}  (g2=0 by symmetry)')

# Assemble the load vector: direction=1, side=1 → v-max = outer arc (r = Rex)
integrator = PatchIntegrator(patch_k, basis_u, basis_v, mat)
f = integrator.integrate_boundary_load(direction=1, side=1, traction=kt)

# Analytic sum of x-DOF loads = Rex * Tx/2 * ∫_0^{π/2} (2cosθ - R2*(2cosθ+3cos3θ) + 3R4*cos3θ) dθ
# Using ∫cosθ dθ = 1, ∫cos3θ dθ = -1/3 on [0,π/2]:
R2 = (Rin/Rex)**2;  R4 = (Rin/Rex)**4
analytic_fx = Rex * Tx/2 * (2 - R2*(2 + 3*(-1/3)) + 3*R4*(-1/3))
print(f'f shape: {f.shape},  ||f|| = {np.linalg.norm(f):.6f}')
print(f'Sum of x-DOF loads: {f[0::2].sum():.6f}  (analytic: {analytic_fx:.4f})')

## Symmetry boundary conditions (Dirichlet)

- `direction=0, side=0` → $\theta = 0$ edge (x-axis): constrain $u_y = 0$ on every CP
- `direction=0, side=1` → $\theta = \pi/2$ edge (y-axis): constrain $u_x = 0$ on every CP

In [ ]:
# Boundary CP local flat indices
bcp_bottom = patch_k.boundary_control_points(0, 0)  # theta=0, constrain DOF 1 (uy)
bcp_left   = patch_k.boundary_control_points(0, 1)  # theta=pi/2, constrain DOF 0 (ux)

constrained_dofs = set()
for cp in bcp_bottom:
    constrained_dofs.add(pdm.get_global_dof_indices(cp)[1])  # uy
for cp in bcp_left:
    constrained_dofs.add(pdm.get_global_dof_indices(cp)[0])  # ux

all_dofs  = set(range(n_dof))
free_dofs = sorted(all_dofs - constrained_dofs)

print(f'Constrained DOFs: {len(constrained_dofs)}  (all zero by symmetry)')
print(f'Free DOFs       : {len(free_dofs)}')

# Extract free sub-system and solve
K_free = K[np.ix_(free_dofs, free_dofs)]
f_free = f[free_dofs]

u_free = spla.spsolve(K_free.tocsc(), f_free)

# Reconstruct the full solution vector
u_sol = np.zeros(n_dof)
u_sol[free_dofs] = u_free

print(f'\nSolve done.  ||u|| = {np.linalg.norm(u_sol):.6f}')
print(f'u_x at (Rex, 0) ≈ {u_sol[pdm.get_global_dof_indices(patch_k.boundary_control_points(1, 1)[0])[0]]:.6f}')

## Comparison with the exact Kirsch solution

**Exact displacement (plane stress)**, obtained by integrating the Kirsch (1898) stress field
(see Timoshenko & Goodier, *Theory of Elasticity*, §71–72 for the displacement derivation):

$$
u_x = \frac{T_x}{2E}\!\left[2\!\left(r + \frac{2a^2}{r}\right)\cos\theta
      + (1+\nu)\frac{a^2}{r}\!\left(1 - \frac{a^2}{r^2}\right)\cos 3\theta\right]
$$
$$
u_y = \frac{T_x}{2E}\!\left[-2\!\left(\nu r + \frac{(1-\nu)a^2}{r}\right)\sin\theta
      + (1+\nu)\frac{a^2}{r}\!\left(1 - \frac{a^2}{r^2}\right)\sin 3\theta\right]
$$

where $a = R_{in}$ and $r = \sqrt{x^2+y^2}$.  This formula satisfies $u_y = 0$ on $\theta=0$,
$u_x = 0$ on $\theta = \pi/2$, and the correct far-field $u_x \to T_x x / E$ as $r \to \infty$.

The **error norms** are computed via `ScalarLocalOperator`: a Python subclass that provides
the scalar integrand at one Gauss point, while the C++ `PatchIntegrator` drives the Gauss
loop, the Jacobian, and the NURBS rationalisation.  Two operator instances are used — one
for the $L^2$ term, one for the $H^1$ semi-norm — each called once via
`integrate_scalar_operator(op, u_sol)`.

In [ ]:
def kirsch_displacement(pt):
    """Exact Kirsch displacement (plane stress, Tx=1) at physical point pt=(x,y).

    Derived by integrating the Kirsch strain field (Timoshenko & Goodier §71-72).
    In polar coordinates:
        u_r = T/(2E) [(1-ν)r + (1+ν)a²/r + ((1+ν)r + 4a²/r - (1+ν)a⁴/r³) cos(2θ)]
        u_θ = T/(2E) [-(1+ν)r - 2(1-ν)a²/r - (1+ν)a⁴/r³] sin(2θ)
    """
    x, y  = pt[0], pt[1]
    r     = np.sqrt(x**2 + y**2)
    theta = np.arctan2(y, x)
    a2    = Rin**2
    u_x   = Tx / (2 * E) * (2 * (r + 2*a2/r) * np.cos(theta)
                             + (1 + nu) * a2/r * (1 - a2/r**2) * np.cos(3*theta))
    u_y   = Tx / (2 * E) * (-2 * (nu*r + (1 - nu)*a2/r) * np.sin(theta)
                             + (1 + nu) * a2/r * (1 - a2/r**2) * np.sin(3*theta))
    return np.array([u_x, u_y])


def kirsch_displacement_gradient(pt, h=1e-6):
    """Numerical gradient of exact displacement via central differences. Returns 2x2 matrix.

    An analytic expression could be derived via the chain rule on (r, θ), but adds ~15 lines
    of dense formulas with no measurable benefit: the O(h²) ≈ 1e-12 truncation error of
    central differences with h=1e-6 is orders of magnitude below the FEM discretization
    errors measured in the convergence study (≥ 1e-7 for all mesh sizes considered here).
    """
    ex = np.array([h, 0.0])
    ey = np.array([0.0, h])
    du_dx = (kirsch_displacement(pt + ex) - kirsch_displacement(pt - ex)) / (2*h)
    du_dy = (kirsch_displacement(pt + ey) - kirsch_displacement(pt - ey)) / (2*h)
    return np.column_stack([du_dx, du_dy])  # shape (2, 2): [[du1/dx, du1/dy],[du2/dx,du2/dy]]


# Verify symmetry BCs on exact solution (interior points of [Rin, Rex])
u_at_xaxis = kirsch_displacement(np.array([1.5, 0.0]))
u_at_yaxis = kirsch_displacement(np.array([0.0, 1.5]))
print(f'Exact u at (1.5, 0  ): {u_at_xaxis}  (uy should be 0)')
print(f'Exact u at (0,   1.5): {u_at_yaxis}  (ux should be 0)')

# Verify far-field: u_x ≈ Tx*x/E as r → ∞
pt_far = np.array([100.0, 0.0])
print(f'Far-field u_x at (100,0): {kirsch_displacement(pt_far)[0]:.5f}  (expected {Tx*100/E:.5f})')

In [ ]:
class L2ErrorOperator(ScalarLocalOperator):
    """Integrand |u_h - u_ex|^2 for the L2 error norm."""

    def __init__(self, u_exact_fn):
        super().__init__()
        self.u_exact_fn = u_exact_fn

    def compute_scalar_integrand(self, R, dRdx, dRdy, physical_point, u_local):
        R    = np.asarray(R)
        u_x  = np.asarray(u_local)[0::2]
        u_y  = np.asarray(u_local)[1::2]
        u_ex = self.u_exact_fn(physical_point)
        return (R @ u_x - u_ex[0])**2 + (R @ u_y - u_ex[1])**2


class H1SemiNormErrorOperator(ScalarLocalOperator):
    """Integrand |∇u_h - ∇u_ex|^2 for the H1 semi-norm error."""

    def __init__(self, grad_exact_fn):
        super().__init__()
        self.grad_exact_fn = grad_exact_fn

    def compute_scalar_integrand(self, R, dRdx, dRdy, physical_point, u_local):
        dRdx = np.asarray(dRdx)
        dRdy = np.asarray(dRdy)
        u_x  = np.asarray(u_local)[0::2]
        u_y  = np.asarray(u_local)[1::2]
        grad_ex = self.grad_exact_fn(physical_point)   # shape (2, 2)
        return (
            (dRdx @ u_x - grad_ex[0, 0])**2 +
            (dRdy @ u_x - grad_ex[0, 1])**2 +
            (dRdx @ u_y - grad_ex[1, 0])**2 +
            (dRdy @ u_y - grad_ex[1, 1])**2
        )


integrator = PatchIntegrator(patch_k, basis_u, basis_v, mat)

err_L2_sq = integrator.integrate_scalar_operator(
    L2ErrorOperator(kirsch_displacement), u_sol)
err_H1_sq = integrator.integrate_scalar_operator(
    H1SemiNormErrorOperator(kirsch_displacement_gradient), u_sol)

err_L2       = np.sqrt(err_L2_sq)
err_H1       = np.sqrt(err_H1_sq)
err_H1_total = np.sqrt(err_L2_sq + err_H1_sq)

print(f'L2 error       : {err_L2:.4e}')
print(f'H1 semi-norm   : {err_H1:.4e}')
print(f'H1 total norm  : {err_H1_total:.4e}')

## Visualisation

Displacement magnitude field compared to the exact Kirsch solution.

In [ ]:
# Evaluate FE solution and exact solution on a grid of evaluation points
n_pts = 60
u_par = np.linspace(0.01, 0.99, n_pts)   # angular parameter
v_par = np.linspace(0.01, 0.99, n_pts)   # radial parameter
UU, VV = np.meshgrid(u_par, v_par)
params = np.column_stack([UU.ravel(), VV.ravel()])   # shape (n_pts^2, 2)

su_k, sv_k = patch_k.tensor.components
spans_arr = np.array(
    [[su_k.find_span(u), sv_k.find_span(v)] for u, v in params], dtype=np.int32)
phys_pts = patch_k.evaluate_patch_nd_omp(spans_arr, params)  # shape (n_pts^2, 2)

# Exact displacement magnitude at each physical point
u_ex_mag = np.array([np.linalg.norm(kirsch_displacement(p)) for p in phys_pts])

# FE displacement via PatchEvaluator: OMP-parallel, handles NURBS rationalisation.
# Returns shape (n_pts^2, 2) — columns are [u_x, u_y] at each evaluation point.
evaluator = PatchEvaluator(patch_k)
u_h = evaluator.evaluate_solution(params, u_sol)
u_h_mag = np.linalg.norm(u_h, axis=1)

# Reshape flat arrays into 2-D grids for plotting
X    = phys_pts[:, 0].reshape(n_pts, n_pts)
Y    = phys_pts[:, 1].reshape(n_pts, n_pts)
U_ex = u_ex_mag.reshape(n_pts, n_pts)
U_h  = u_h_mag.reshape(n_pts, n_pts)

vmin = min(U_ex.min(), U_h.min())
vmax = max(U_ex.max(), U_h.max())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Exact
c0 = axes[0].pcolormesh(X, Y, U_ex, vmin=vmin, vmax=vmax, cmap='viridis', shading='auto')
axes[0].set_aspect('equal')
axes[0].set_title('Exact $|\\mathbf{u}|$ (Kirsch)')
plt.colorbar(c0, ax=axes[0])

# FE
c1 = axes[1].pcolormesh(X, Y, U_h, vmin=vmin, vmax=vmax, cmap='viridis', shading='auto')
axes[1].set_aspect('equal')
axes[1].set_title('FE $|\\mathbf{u}_h|$')
plt.colorbar(c1, ax=axes[1])

# Error
err_pt = np.abs(U_h - U_ex)
c2 = axes[2].pcolormesh(X, Y, err_pt, cmap='Reds', shading='auto')
axes[2].set_aspect('equal')
axes[2].set_title('Pointwise error $|\\,|\\mathbf{u}_h| - |\\mathbf{u}|\\,|$')
plt.colorbar(c2, ax=axes[2])

for ax in axes:
    ax.set_xlabel('x')
    ax.set_ylabel('y')

p_u = su_k.degree
p_v = sv_k.degree
plt.suptitle(
    f'Kirsch benchmark — k-refinement degree {p_u}×{p_v}, {n_cp} CPs\n'
    f'$L^2$ error = {err_L2:.2e},  $H^1$ semi-norm = {err_H1:.2e}',
    fontsize=11)
plt.tight_layout()
plt.show()

## Convergence study — k-refinement

For each target degree $p \in \{2, 3, 4, 5\}$ and each mesh size
$n \in \{1, 2, 4, 8, 16, 32\}$ elements per direction, the patch is rebuilt from the
base NURBS geometry and then k-refined (degree elevation followed by uniform knot
insertion) so that both parametric directions share the same degree $p$ and the
same number of elements $n$.

The expected convergence rates for k-refinement of degree $p$ (slope on a log–log
plot against total element count $N_{el} = n^2$):

| Norm | Rate | Log–log slope vs $N_{el}$ |
|------|------|---------------------------|
| $L^2$ displacement | $\mathcal{O}(h^{p+1})$ | $-(p+1)/2$ |
| $H^1$ semi-norm (energy) | $\mathcal{O}(h^{p})$ | $-p/2$ |

These rates are established in **Bazilevs, Beirão da Veiga, Cottrell, Hughes & Sangalli
(2006)**, *Isogeometric analysis: approximation, stability and error estimates for
h-refined meshes*, Mathematical Models and Methods in Applied Sciences, **16**(07),
1031–1090 (Theorems 3.1–3.2 for B-spline/NURBS spaces on mapped domains).

**Note on domain size**: $R_{ex}=2$ is chosen so that the analytic singularity of
$1/r(v)$ in the parametric $v$-direction sits at $v=-1$ (distance 1 from $[0,1]$),
well outside the domain.  A larger $R_{ex}$ (e.g. 4) pushes the singularity to
$v \approx -1/(R_{ex}-1)$, close to 0, strongly extending the pre-asymptotic
regime for high degrees.

In [ ]:
degrees = [2, 3, 4, 5]
n_elems = [1, 2, 4, 8, 16, 32]

# conv_results[p] = list of (n_el_total, err_L2, err_H1)
conv_results = {p: [] for p in degrees}

for p in degrees:
    for n in n_elems:
        # --- rebuild base geometry from scratch ---
        mgr_c = ControlPointManager(dim=2)
        mgr_c.add_point([Rin, 0.0], w=1.0)
        mgr_c.add_point([Rin, Rin], w=1.0 / np.sqrt(2.0))
        mgr_c.add_point([0.0, Rin], w=1.0)
        mgr_c.add_point([Rex, 0.0], w=1.0)
        mgr_c.add_point([Rex, Rex], w=1.0 / np.sqrt(2.0))
        mgr_c.add_point([0.0, Rex], w=1.0)

        su_c = BSpline(2, np.array([0., 0., 0., 1., 1., 1.]))
        sv_c = BSpline(1, np.array([0., 0., 1., 1.]))
        patch_c = Patch(BSplineSurface(su_c, sv_c), mgr_c, list(range(6)), [3, 2])

        # k-refinement: elevate both directions to degree p, then subdivide
        if p > 2:
            PRefiner(direction=0, n_elevations=p - 2).refine(patch_c)
        PRefiner(direction=1, n_elevations=p - 1).refine(patch_c)

        n_levels = n.bit_length() - 1
        if n_levels > 0:
            SubdivisionRefiner(direction=0, n_levels=n_levels).refine(patch_c)
            SubdivisionRefiner(direction=1, n_levels=n_levels).refine(patch_c)

        # --- DOF manager ---
        n_cp_c = patch_c.n_cp
        gm_c   = GlobalDOFManager([2] * n_cp_c)
        pdm_c  = PatchDOFManager(2, list(range(n_cp_c)), gm_c)
        pk_c   = Patch(patch_c.tensor, patch_c.cp_manager,
                       list(patch_c.global_indices), list(patch_c.local_shape), pdm_c)

        su_kc, sv_kc = pk_c.tensor.components
        bu_c  = IGABasis1D.build(su_kc, su_kc.degree + 1)
        bv_c  = IGABasis1D.build(sv_kc, sv_kc.degree + 1)
        mat_c = MaterialProperties(E=E, nu=nu)

        # --- stiffness and load ---
        K_c = PatchIntegrator(pk_c, bu_c, bv_c, mat_c).integrate_stiffness()
        f_c = PatchIntegrator(pk_c, bu_c, bv_c, mat_c).integrate_boundary_load(
            direction=1, side=1, traction=KirschTraction(Rin, Tx))

        # --- Dirichlet BCs (symmetry) ---
        constrained_c = set()
        for cp in pk_c.boundary_control_points(0, 0):
            constrained_c.add(pdm_c.get_global_dof_indices(cp)[1])
        for cp in pk_c.boundary_control_points(0, 1):
            constrained_c.add(pdm_c.get_global_dof_indices(cp)[0])

        n_dof_c = 2 * n_cp_c
        free_c  = sorted(set(range(n_dof_c)) - constrained_c)
        u_sol_c = np.zeros(n_dof_c)
        u_sol_c[free_c] = spla.spsolve(
            K_c[np.ix_(free_c, free_c)].tocsc(), f_c[free_c])

        # --- L2 and H1 errors ---
        integ_c = PatchIntegrator(pk_c, bu_c, bv_c, mat_c)
        eL2 = np.sqrt(integ_c.integrate_scalar_operator(
            L2ErrorOperator(kirsch_displacement), u_sol_c))
        eH1 = np.sqrt(integ_c.integrate_scalar_operator(
            H1SemiNormErrorOperator(kirsch_displacement_gradient), u_sol_c))

        conv_results[p].append((n**2, eL2, eH1))
        print(f'p={p}, n={n:2d} ({n**2:5d} el): {n_cp_c:4d} CPs, L2={eL2:.3e}, H1={eH1:.3e}')

# ── Plot ──────────────────────────────────────────────────────────────────────
colors  = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
markers = ['o', 's', '^', 'D']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax_idx, (norm_label, err_idx, slope_fn) in enumerate([
        ('$L^2$ error',      1, lambda p: -(p + 1) / 2),
        ('$H^1$ semi-norm',  2, lambda p: -p / 2),
]):
    ax = axes[ax_idx]
    all_errs, all_ns = [], [n**2 for n in n_elems]

    for i, p in enumerate(degrees):
        ns   = [r[0]        for r in conv_results[p]]
        errs = [r[err_idx]  for r in conv_results[p]]
        all_errs.extend(errs)
        ax.loglog(ns, errs, marker=markers[i], color=colors[i],
                  label=f'degree {p}', linewidth=1.5, markersize=6)

    ax.set_xlim(min(all_ns) * 0.6, max(all_ns) * 1.8)
    ax.set_ylim(min(all_errs) * 0.3, max(all_errs) * 3.0)

    for i, p in enumerate(degrees):
        slope    = slope_fn(p)
        n0       = float(conv_results[p][0][0])
        n1, e1   = float(conv_results[p][-1][0]), conv_results[p][-1][err_idx]
        x_ref    = np.array([n0, n1])
        c        = e1 / (n1 ** slope)          # anchor on the last simulation point
        ax.loglog(x_ref, c * x_ref**slope, '--', color=colors[i], alpha=0.4, linewidth=1)
        ax.text(n1 * 1.05, e1, f'{slope:.1f}', color=colors[i], va='center', fontsize=9)

    ax.set_xlabel('Total number of elements $N_{el} = n^2$')
    ax.set_ylabel(norm_label)
    ax.legend(title='Degree (u & v)', loc='upper right', fontsize=9)
    ax.grid(True, which='both', alpha=0.3)

fig.suptitle(
    f'Kirsch benchmark — k-refinement convergence\n'
    f'($E={E:.0f}$, $\\nu={nu}$, $R_{{in}}={Rin}$, $R_{{ex}}={Rex}$)',
    fontsize=12)
plt.tight_layout()
plt.show()